## Metrics on mapping with batch set as sample, kl1, original gene counts

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.append("../../existing_metric_comparison/")
sys.path.append("../../../src/")
sys.path.append("../../utils/HLCA")
import data_cleanup

# import metrics_lib as metrics
import os
import mapqc

In [ ]:
sc.set_figure_params(figsize=(5, 5))

## Set global variables

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
kl = "1.0"
use_old_counts = True
if use_old_counts == True:
    suffix = "original_counts_"
else:
    suffix = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_{suffix}batch_{batch_variable}_kl{kl}_latent.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_counts.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/benchmark"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/mapqc"

In [ ]:
if not os.path.exists(path_results_dir):
    if os.path.exists(os.path.dirname(path_results_dir)):
        print(f"Creating directory {path_results_dir}")
        os.mkdir(RESULT_DIR)
    else:
        print("Parent directory of path_results_dir does not exist. Create it first.")

In [ ]:
path_emb = "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_Sheppard_2020_original_counts_batch_sample_kl1.0_latent.h5ad"

In [ ]:
path_counts = "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_Sheppard_2020_counts.h5ad"

## load and prep data

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata_full = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

import mapqc results if wanted (for comparison):

In [ ]:
path_mapqc_results_dir

In [ ]:
results_healthy = mapqc.load(
    os.path.join(path_mapqc_results_dir, "control_edistance_k500.pkl")
)
results_disease = mapqc.load(
    os.path.join(path_mapqc_results_dir, "case_edistance_k500.pkl")
)

## run metrics

In [ ]:
query_condition_1 = "Healthy"

In [ ]:
adata_1 = adata_full[
    (adata_full.obs.lung_condition == query_condition_1)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

subset query for metric functions below

In [ ]:
adata_query_1 = adata_1[adata_1.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_1 = metrics.calculate_scib_metrics(
    adata_1, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_1 = metrics.calculate_symphony_metrics(
    adata_1,
    adata_query_1,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_1 = metrics.calculate_other_metrics(
    adata_1, adata_query_1, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_1.obs["lisi_batch"] = scib_results_1["lisi_batch"]["per_cell"]
adata_1.obs["kbet_batch"] = scib_results_1["kbet_batch"]["per_cell"]
adata_1.obs["lisi_query_donor"] = symphony_results_1["lisi_query_donor"]["per_cell"]
adata_1.obs["wiq_knn_corr"] = symphony_results_1["wiq_knn_corr"]["per_cell"]
adata_1.obs["cluster_preservation_score"] = other_results_1[
    "cluster_preservation_score"
]["per_cell"]

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(path_results_dir, f"alt_metrics_{query_condition_1.lower()}.csv"))

## Visualize 

In [ ]:
sc.pl.umap(
    adata_1, color="r_or_q", frameon=False, palette={"r": "grey", "q": "dodgerblue"}
)

Load results from above (stored in csv file)

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(path_results_dir, f"alt_metrics_{query_condition_1.lower()}.csv"),
    index_col="Unnamed: 0",
).loc[
    adata_1.obs.index, :
]

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_1.obs.loc[adata_1.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_1, color=metric)  # ,vmax='p99')

Quantification/visualization different metrics:

In [ ]:
hist_kwargs = {
    "wiq_knn_corr": {"range": (-1, 1), "bins": 10},
    "cluster_preservation_score": {"bins": 10},
    "lisi_query_donor": {"bins": 10},
    "lisi_batch": {"bins": 10},
    "kbet_batch": {"range": (0, 1)},
}
xlabels = {
    "wiq_knn_corr": "Within-query\nk-NN-correlation (↑)",
    "cluster_preservation_score": "Cluster preservation\nscore (↑)",
    "lisi_query_donor": "Query donor LISI (↑)",
    "lisi_batch": "Batch LISI (↑)",
    "kbet_batch": "kBET (↑)",
}
results = {}

In [ ]:
alt_metrics_to_include = [
    "wiq_knn_corr",
    "cluster_preservation_score",
    "lisi_query_donor",
    "lisi_batch",
    "kbet_batch",
]
fignumber = 0
n_figures = len(alt_metrics_to_include) + 2
n_cols = n_figures
n_rows = int(np.ceil(n_figures / n_cols))
fig = plt.figure(figsize=(2.5 * n_cols, 3 * n_rows))
fig_count = 1

# start with mapqc scores, x (score) range 0-10
ax = fig.add_subplot(n_rows, n_cols, fig_count)
mapqc_scores = results_healthy.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy.adata.obs.mapqc_score)
]
xmax = 10
# check that we are not ignoring data
if np.percentile(mapqc_scores, 99) > xmax:
    raise ValueError("You are ignoring more than 1% of your data. Change your xmax.")
ax.hist(
    mapqc_scores,
    bins=10,
    range=(0, xmax),
    rwidth=0.9,
)  # density=True,
ax.set_xlabel("MapQC score (↓)")
ax.set_ylabel("N cells")  # Fraction of cells")
ax.spines[["top", "right"]].set_visible(False)
ax.vlines(x=2, ymin=0, ymax=13000, color="red", linestyle="--")
plt.grid(False)
fig_count += 1

# now plot rest of mapqc range: 10-100
ax = fig.add_subplot(n_rows, n_cols, fig_count)
mapqc_scores = results_healthy.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy.adata.obs.mapqc_score)
]
xmin = 10
xmax = 100
# check that we are not ignoring data
if np.percentile(mapqc_scores, 99) > xmax:
    raise ValueError("You are ignoring more than 1% of your data. Change your xmax.")
ax.hist(mapqc_scores, bins=10, range=(xmin, xmax), rwidth=0.9)  # density=True,
ax.set_xlabel("MapQC score (↓)")
ax.spines[["top", "right"]].set_visible(False)
# plot line for cutting of figure
ax.vlines(x=9.5, ymin=0, ymax=13000, color="blue", linestyle="--")
plt.grid(False)
results["mapqc"] = {"mean": np.mean(mapqc_scores), "median": np.median(mapqc_scores)}
fig_count += 1

# then loop through other metrics
for metric in alt_metrics_to_include:
    ax = fig.add_subplot(n_rows, n_cols, fig_count)
    metric_vals = adata_1.obs[metric][adata_1.obs.r_or_q == "q"]
    ax.hist(metric_vals, rwidth=0.9, **hist_kwargs[metric])
    ax.set_ylabel("N cells")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(xlabels[metric])
    ax.yaxis.label.set_visible(
        False
    )  # remove y-axis label, it's the same for all plots
    fig_count += 1
    plt.grid(False)
    results[metric] = {"mean": np.mean(metric_vals), "median": np.median(metric_vals)}
plt.tight_layout()
plt.show()
plt.close()
for metric, res in results.items():
    print(f"{metric}: {res}")

#### Plot together with results from bad mapping:

Load results bad mapping:

In [ ]:
path_mapqc_results_dir_bad_mapping = '/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/Sheppard_2020/cellxgene_counts_batch_dataset_kl2.0/mapqc'
path_results_dir_bad_mapping = '/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/Sheppard_2020/cellxgene_counts_batch_dataset_kl2.0/benchmark'


In [ ]:
results_healthy_bad_mapping = mapqc.load(
    os.path.join(path_mapqc_results_dir_bad_mapping, "control_edistance_k500.pkl")
)

In [ ]:
adata_1.obs[
    [
        "bad_mapping_lisi_batch",
        "bad_mapping_kbet_batch",
        "bad_mapping_lisi_query_donor",
        "bad_mapping_wiq_knn_corr",
        "bad_mapping_cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(path_results_dir_bad_mapping, f"alt_metrics_{query_condition_1.lower()}.csv"),
    index_col="Unnamed: 0",
).loc[
    adata_1.obs.index, :
]

In [ ]:
color_good_mapping = "#4daf4a" # "green"
color_bad_mapping = "#e41a1c" # "red"
alt_metrics_to_include = [
    "wiq_knn_corr",
    "cluster_preservation_score",
    "lisi_query_donor",
    "lisi_batch",
    "kbet_batch",
]
fignumber = 0
n_figures = len(alt_metrics_to_include) + 2
n_cols = n_figures
n_rows = int(np.ceil(n_figures / n_cols))
fig = plt.figure(figsize=(2.5 * n_cols, 3 * n_rows))
fig_count = 1

# start with mapqc scores, x (score) range 0-10
ax = fig.add_subplot(n_rows, n_cols, fig_count)
mapqc_scores = results_healthy.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy.adata.obs.mapqc_score)
]
mapqc_scores_bad_mapping = results_healthy_bad_mapping.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy_bad_mapping.adata.obs.mapqc_score)
]
xmax = 10
# check that we are not ignoring data
if np.percentile(mapqc_scores, 99) > xmax:
    raise ValueError("You are ignoring more than 1% of your data. Change your xmax.")
ax.hist(
    mapqc_scores,
    bins=10,
    range=(0, xmax),
    rwidth=0.9,
    color=color_good_mapping,
    alpha=0.5
    
)  # density=True,
ax.hist(
    mapqc_scores_bad_mapping,
    bins=10,
    range=(0, xmax),
    rwidth=0.9,
    color=color_bad_mapping,
    alpha=0.5
    
)
ax.set_xlabel("MapQC score (↓)")
ax.set_ylabel("N cells")  # Fraction of cells")
ax.spines[["top", "right"]].set_visible(False)
ax.vlines(x=2, ymin=0, ymax=13000, color="red", linestyle="--")
plt.grid(False)
fig_count += 1

# now plot rest of mapqc range: 10-100
ax = fig.add_subplot(n_rows, n_cols, fig_count)
mapqc_scores = results_healthy.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy.adata.obs.mapqc_score)
]
xmin = 10
xmax = 100
# check that we are not ignoring data
if np.percentile(mapqc_scores, 99) > xmax:
    raise ValueError("You are ignoring more than 1% of your data. Change your xmax.")
ax.hist(mapqc_scores, bins=10, range=(xmin, xmax), rwidth=0.9, color=color_good_mapping, alpha=0.5)  # density=True,
ax.hist(mapqc_scores_bad_mapping, bins=10, range=(xmin, xmax), rwidth=0.9, color=color_bad_mapping, alpha=0.5)  # density=True,
ax.set_xlabel("MapQC score (↓)")
ax.spines[["top", "right"]].set_visible(False)
# plot line for cutting of figure
ax.vlines(x=9.5, ymin=0, ymax=13000, color="blue", linestyle="--")
plt.grid(False)
results["mapqc"] = {"mean": np.mean(mapqc_scores), "median": np.median(mapqc_scores)}
fig_count += 1

# then loop through other metrics
for metric in alt_metrics_to_include:
    ax = fig.add_subplot(n_rows, n_cols, fig_count)
    metric_vals = adata_1.obs[metric][adata_1.obs.r_or_q == "q"]
    metric_vals_bad_mapping = adata_1.obs[f"bad_mapping_{metric}"][adata_1.obs.r_or_q == "q"]
    ax.hist(metric_vals, rwidth=0.9, **hist_kwargs[metric], color=color_good_mapping, alpha=0.5)
    ax.hist(metric_vals_bad_mapping, rwidth=0.9, **hist_kwargs[metric], color=color_bad_mapping, alpha=0.5)
    ax.set_ylabel("N cells")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(xlabels[metric])
    ax.yaxis.label.set_visible(
        False
    )  # remove y-axis label, it's the same for all plots
    fig_count += 1
    plt.grid(False)
    results[metric] = {"mean": np.mean(metric_vals), "median": np.median(metric_vals)}
plt.tight_layout()
plt.show()
plt.close()
for metric, res in results.items():
    print(f"{metric}: {res}")

### Query disease (figures not used for paper)

In [ ]:
query_condition_2 = "IPF"

In [ ]:
adata_2 = adata_full[
    (adata_full.obs.lung_condition == query_condition_2)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

In [ ]:
del adata_full

subset query for metric functions below

In [ ]:
adata_query_2 = adata_2[adata_2.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_2 = metrics.calculate_scib_metrics(
    adata_2, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_2 = metrics.calculate_symphony_metrics(
    adata_2,
    adata_query_2,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_2 = metrics.calculate_other_metrics(
    adata_2, adata_query_2, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_2.obs["lisi_batch"] = scib_results_2["lisi_batch"]["per_cell"]
adata_2.obs["kbet_batch"] = scib_results_2["kbet_batch"]["per_cell"]
adata_2.obs["lisi_query_donor"] = symphony_results_2["lisi_query_donor"]["per_cell"]
adata_2.obs["wiq_knn_corr"] = symphony_results_2["wiq_knn_corr"]["per_cell"]
adata_2.obs["cluster_preservation_score"] = other_results_2[
    "cluster_preservation_score"
]["per_cell"]

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(path_results_dir, f"alt_metrics_{query_condition_2.lower()}.csv"))

## Visualize 

Run Load and prep embeddings and set global variables first

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(path_results_dir, f"alt_metrics_{query_condition_2.lower()}.csv"),
    index_col="Unnamed: 0",
)

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_2.obs.loc[adata_2.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_2, color=metric)  # ,vmax='p99')